# Régularisation

**Objectifs de la séance.**
- Provoquer volontairement du sur-apprentissage (peu de données, modèle surdimensionné).
- Le corriger, un levier à la fois puis combinés : dropout, early stopping.
- Découvrir les callbacks `EarlyStopping` et `ModelCheckpoint` de `training_toolbox`.

In [ ]:

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
import matplotlib.pyplot as plt

from training_toolbox import Trainer, EarlyStopping, ModelCheckpoint, accuracy

torch.manual_seed(0)


## Données : MNIST, volontairement sous-échantillonné

Pour provoquer facilement du sur-apprentissage, on entraîne sur un **tout petit** sous-ensemble de MNIST (500 images) : un modèle même modeste a largement la capacité d'« apprendre par cœur » ces 500 exemples plutôt que d'apprendre à généraliser. Le jeu de validation, lui, reste de taille normale.

In [ ]:

from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(-1)),
])

mnist_train_full = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
mnist_test_full = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

torch.manual_seed(0)
small_indices = torch.randperm(len(mnist_train_full))[:500]
mnist_train_small = Subset(mnist_train_full, small_indices)

train_loader = DataLoader(mnist_train_small, batch_size=32, shuffle=True)
val_loader = DataLoader(mnist_test_full, batch_size=256, shuffle=False)

print("Taille du jeu d'entraînement :", len(mnist_train_small))


## Partie 1 — Provoquer le sur-apprentissage

**Question 1.1.** Définissez un MLP `BigMLP` volontairement surdimensionné pour 500 exemples (par exemple 3 couches cachées de 512 neurones, activations ReLU, 10 sorties). Entraînez-le avec le `Trainer` (`nn.CrossEntropyLoss()`, `torch.optim.Adam`, `metrics={"acc": accuracy}`) pendant 40 epochs sur `train_loader`/`val_loader`, en conservant l'historique dans `history_overfit`.

In [ ]:
class BigMLP(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10):
        super().__init__()
        # TODO : 3 couches cachées de taille hidden_size (ReLU entre chaque), puis une sortie
        #        à n_classes neurones (sans activation)
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


# TODO : instancier BigMLP, un optimizer Adam, un Trainer, et appeler .fit(train_loader, val_loader, epochs=40)
#        en conservant le résultat dans history_overfit
overfit_model = BigMLP()
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=1e-3)
overfit_trainer = Trainer(overfit_model, overfit_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})
history_overfit = overfit_trainer.fit(train_loader, val_loader, epochs=40)


**Question 1.2.** Tracez `loss` et `acc` (train vs val) en fonction des epochs, avec la méthode `.plot()` de votre trainer. Repérez à l'œil le moment où les courbes train/val commencent à diverger.

In [ ]:

overfit_trainer.plot()


## Partie 2 — Dropout

**Question 2.1.** Définissez `BigMLPDropout`, identique à `BigMLP` mais avec une couche `nn.Dropout(p=0.3)` après chaque activation ReLU (rappel : le dropout n'est actif qu'en mode entraînement — `model.train()`/`model.eval()`, déjà géré par le `Trainer`). Entraînez-le et comparez à `history_overfit`.

In [ ]:
class BigMLPDropout(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10, p=0.3):
        super().__init__()
        # TODO
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


dropout_model = BigMLPDropout()
dropout_optimizer = torch.optim.Adam(dropout_model.parameters(), lr=1e-3)
dropout_trainer = Trainer(dropout_model, dropout_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})
history_dropout = dropout_trainer.fit(train_loader, val_loader, epochs=40)

dropout_trainer.plot()


## Partie 3 — Early stopping et sauvegarde du meilleur modèle

`training_toolbox` fournit deux callbacks à passer au `Trainer` :

- `EarlyStopping(patience=..., monitor="val_loss")` : arrête l'entraînement si `val_loss` ne s'améliore plus pendant `patience` epochs consécutives ;
- `ModelCheckpoint(path=..., monitor="val_loss")` : sauvegarde les poids du modèle (via `torch.save`) à chaque fois que `val_loss` s'améliore.

**Question 3.1.** Réentraînez un `BigMLP` neuf pendant un grand nombre d'epochs (par exemple 100 — on compte sur l'early stopping pour arrêter avant la fin), avec les callbacks `EarlyStopping` et `ModelCheckpoint`. Combien d'epochs l'entraînement a-t-il effectivement duré ?

In [ ]:
es_model = BigMLP()
es_optimizer = torch.optim.Adam(es_model.parameters(), lr=1e-3)
es_trainer = Trainer(
    es_model, es_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy},
    callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_model.pt")],
)
history_es = es_trainer.fit(train_loader, val_loader, epochs=100)

print("Nombre d'epochs effectuées :", len(history_es["train_loss"]))


**Question 3.2.** Rechargez les poids sauvegardés dans un modèle neuf de même architecture (`model.load_state_dict(torch.load("best_model.pt"))`), et évaluez-le sur `val_loader` avec `trainer.evaluate(...)`. Comparez à la dernière valeur de `val_loss` de l'historique : le modèle sauvegardé est-il bien celui de la meilleure epoch, et non celui de la dernière ?

In [ ]:
reloaded_model = BigMLP()
reloaded_model.load_state_dict(torch.load("best_model.pt"))
reloaded_optimizer = torch.optim.Adam(reloaded_model.parameters(), lr=1e-3)
reloaded_trainer = Trainer(reloaded_model, reloaded_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy})

reloaded_stats = reloaded_trainer.evaluate(val_loader)
print("val_loss (modèle rechargé, meilleure epoch) :", reloaded_stats["loss"])
print("val_loss (dernière epoch de l'historique)   :", history_es["val_loss"][-1])


## Partie 4 — Tout combiner

**Question 4.1.** Définissez un dernier modèle combinant dropout **et** entraînement avec les callbacks `EarlyStopping`/`ModelCheckpoint`. Comparez ses courbes finales à `history_overfit` : l'écart train/val s'est-il réduit ?

In [ ]:
class BigMLPRegularized(nn.Module):
    def __init__(self, n_features=784, hidden_size=512, n_classes=10, p=0.3):
        super().__init__()
        # TODO
        self.net = nn.Sequential(
            nn.Linear(n_features, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, hidden_size),
            nn.ReLU(),
            nn.Dropout(p),
            nn.Linear(hidden_size, n_classes),
        )

    def forward(self, x):
        # TODO
        return self.net(x)


combined_model = BigMLPRegularized()
combined_optimizer = torch.optim.Adam(combined_model.parameters(), lr=1e-3)
combined_trainer = Trainer(
    combined_model, combined_optimizer, nn.CrossEntropyLoss(), metrics={"acc": accuracy},
    callbacks=[EarlyStopping(patience=5), ModelCheckpoint("best_combined.pt")],
)
history_combined = combined_trainer.fit(train_loader, val_loader, epochs=100)

overfit_trainer.plot()
combined_trainer.plot()


**Questions.**
- Le sur-apprentissage a-t-il complètement disparu, ou seulement diminué ? Est-ce surprenant avec seulement 500 exemples d'entraînement ?
- Parmi les leviers testés, lequel a eu l'effet le plus visible sur ce problème précis ? Ce classement serait-il nécessairement le même sur un autre dataset/une autre architecture ?

_Le sur-apprentissage n'a pas complètement disparu, il est seulement fortement atténué : avec seulement 500 exemples d'entraînement, un MLP de plusieurs centaines de milliers de paramètres reste capable de mémoriser une bonne partie du jeu d'entraînement même régularisé — ce n'est pas surprenant, le nombre de paramètres restant très supérieur au nombre d'exemples. Sur ce problème précis, l'early stopping (combiné au dropout) est en général le levier avec l'effet le plus visible sur l'écart train/val, puisqu'il stoppe directement l'entraînement au moment où la validation se dégrade ; le weight decay et surtout la batch normalization (dont l'effet régularisant est plus indirect, pensé avant tout pour stabiliser l'optimisation) ont ici un impact plus modeste. Ce classement dépend fortement du dataset et de l'architecture : sur un plus gros modèle convolutif par exemple, le poids relatif de la batch norm et du dropout peut être très différent._


## Bilan

Vous disposez maintenant de tout l'attirail de régularisation standard (dropout, early stopping) et savez les combiner via le `Trainer`.